# Hydrograph response: six measures of event change

Primary results use the **completed reruns and a shared 0.5% response window**, across 15 examples, 2,202 model states and 2,187 parent–child comparisons. The 1% and 2% windows are retained as sensitivity checks.

Each example has one **fixed minimum discharge baseline** and identical analysis start/end times for every state. The percentage is a timing tolerance around the initial/final plateaus; it is **not** the baseline discharge. All plots use Plotly; times are UTC.

These are observations of the routed hydrographs. Geometry and routing parameters explain them in a separate analysis.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, clear_output

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "RAPID/src/rapid_tools").is_dir())
sys.path.insert(0, str(ROOT / "RAPID/src"))
from rapid_tools.hydrograph_response import (
    REVIEW, OUT, METRICS, calculate, load, select_pair,
    threshold_figure, metric_figure, comparison_table, overview_figure, sensitivity_figure,
)
pio.renderers.default = "plotly_mimetype+notebook"
# Recalculation reads existing model outputs; it does not rerun routing.
RECALCULATE = False
if RECALCULATE or not (OUT / "responses.parquet").exists():
    calculate()
states, responses = load("0.5%")  # Verifies current definitions, calculation source and table hashes.
manifest = json.loads((OUT / "manifest.json").read_text())
display(pd.DataFrame([{
    "examples": states.example.nunique(), "model versions": len(states),
    "partial-collapse comparisons": int(responses.comparison.eq("partial collapse").sum()),
    "single-centerline comparisons": int(responses.comparison.eq("single centerline").sum()),
    "complete six-response rows": int(responses[list(METRICS)].notna().all(axis=1).sum()),
}]))

,examples,model versions,partial-collapse comparisons,single-centerline comparisons,complete six-response rows
0,15,2202,2172,15,2187


## 1. Baseline and analysis window

Let $R=Q_{\mathrm{input,max}}-Q_{\mathrm{input,min}}$ over the selected input event. The primary timing tolerance is $\epsilon=0.005R$.

- **Start:** earliest modeled departure satisfying $|Q-Q_{\mathrm{initial}}|>\epsilon$ for 30 consecutive saved minutes, across all states. The 30-minute persistence is an operational convention.
- **End:** first saved time after the last post-input departure satisfying $|Q-Q_{\mathrm{final}}|>\epsilon$ across any state. All remaining saved output is checked for later excursions.
- **Shared baseline $B$:** minimum across every completed model curve after the input start, including the initial and final equilibrium levels. Gauges are excluded. This same baseline is held fixed for the 0.5%, 1% and 2% calculations.

On the common window $[t_a,t_b]$, $e(t)=Q(t)-B$. Endpoint excess is retained: it need not be zero. No below-baseline discharge is clipped, and no artificial taper is added. Spin-up, the 96-hour confirmation period and unused tail padding are excluded.

This is a finite-window excess-volume description, **not physical baseflow separation**. A final plateau above $B$ still contributes volume inside the selected window. Volume, centroid, central-80% duration and asymmetry can therefore remain window-sensitive.

The saved one-minute interval means are reconstructed linearly; volume, moments and quantiles are integrated analytically between those nodes. Peak timing uses the first saved maximum in the common window. Every selected peak is checked against the full post-input-start model maximum. The source-clock caveat for 21/22 remains.

In [2]:
definitions = states.groupby("example").agg(
    baseline_cms=("baseline_cms", "first"), tolerance_cms=("tolerance_cms", "first"),
    onset_utc=("rise_utc", "first"), end_utc=("fall_utc", "first"), versions=("state", "size"),
)
display(definitions)

,baseline_cms,tolerance_cms,onset_utc,end_utc,versions
example,,,,,
02,946.986290,10.511373,2025-07-31T17:05:30+00:00,2025-08-04T18:07:30+00:00,37
03,1520.700000,5.553565,2023-02-15T03:07:30+00:00,2023-02-20T22:23:30+00:00,42
05,328.700000,3.579500,2024-01-04T10:56:30+00:00,2024-01-16T18:32:30+00:00,53
13,647.900000,1.076000,2024-10-03T18:07:30+00:00,2024-10-10T20:27:30+00:00,68
15,1050.000000,5.550000,2026-04-01T21:33:30+00:00,2026-04-13T11:07:30+00:00,264
21,8135.052740,31.998850,2025-07-27T11:45:30+00:00,2025-08-08T02:11:30+00:00,32
22,18825.528487,7.925079,2025-08-17T20:38:30+00:00,2025-08-27T10:13:30+00:00,410
37,1469.833117,6.029129,2014-06-26T01:03:30+00:00,2014-07-01T15:21:30+00:00,398
41,4729.857276,14.578457,2024-07-08T12:23:30+00:00,2024-08-01T18:05:30+00:00,184


## 2. Inspect the baseline and shared window

Choose an example and optional child. The plot shows the parent, single centerline and selected child, with both gauges in original discharge units. Dotted gauge segments flag long sampling intervals.

**Full** shows the saved simulation; **event** shows the primary shared 0.5% window; **peak** focuses on the displayed crests. The horizontal line is the minimum baseline. Vertical lines mark the common start/end; curve markers retain the actual discharge at those times, rather than forcing endpoints to zero.

The controls require `ipywidgets`; the explicit plotting cell below also works without widgets.

In [3]:
import ipywidgets as widgets
examples = sorted(states.example.unique())
example_picker = widgets.Dropdown(options=examples, value="46", description="Example:")
child_picker = widgets.Dropdown(description="Child:", layout=widgets.Layout(width="650px"))
view_picker = widgets.Dropdown(options=["full", "event", "peak"], value="event", description="View:")
show_gauges = widgets.Checkbox(value=True, description="Show both gauges")
threshold_output = widgets.Output()

def refresh_threshold_plot(*_):
    if child_picker.value is None:
        return
    with threshold_output:
        clear_output(wait=True)
        display(threshold_figure(states, example_picker.value, child_picker.value,
                                 view_picker.value, show_gauges.value))

def update_child_choices(*_):
    g = states[(states.example == example_picker.value) & (states.state != "S000_base")]
    child_picker.options = [("Single centerline" if r.state == "fully_collapsed" else r.aliases, r.state)
                            for r in g.itertuples()]
    child_picker.value = "fully_collapsed"

update_child_choices()
example_picker.observe(update_child_choices, names="value")
child_picker.observe(refresh_threshold_plot, names="value")
view_picker.observe(refresh_threshold_plot, names="value")
show_gauges.observe(refresh_threshold_plot, names="value")
display(widgets.VBox([example_picker, child_picker, widgets.HBox([view_picker, show_gauges]), threshold_output]))
refresh_threshold_plot()

## 3. Select a worked parent–child comparison

Change `EXAMPLE` and `CHILD`, then rerun this cell and the six metric cells below. Use `fully_collapsed` for the single centerline, or an exact state ID/alias. The original motivating comparison is **46 / CB006_bubble_101**; that alias is resolved to its canonical saved state.

All changes use **child minus parent**, or **100 × (child / parent − 1)** for percentages. A positive value means later, higher, wider, more late-weighted, or more excess volume, respectively. It does not always mean “slower” or “worse.”

In [4]:
EXAMPLE = "46"
CHILD = "CB006_bubble_101"  # or "fully_collapsed"
parent, child = select_pair(states, EXAMPLE, CHILD)
selected = responses[(responses.example == parent.example) & (responses.state == child.state)].iloc[0]
print(f"Example {EXAMPLE}: S000_base → {child.aliases}")
print(f"Shared minimum baseline: {parent.baseline_cms:.6f} m³/s")
display(comparison_table(responses, states, EXAMPLE, CHILD))
threshold_figure(states, EXAMPLE, CHILD, view="event").show()

Example 46: S000_base → S139_unit_13 | CB006_bubble_101
Shared minimum baseline: 813.498455 m³/s


,response,parent,child,change,change_unit
0,Peak-time shift,2024-09-03T12:38:30+00:00,2024-09-03T12:42:30+00:00,0.066667,hours
1,Peak-excess change,1059.167975,1064.756179,0.527603,%
2,Excess-volume centroid shift,2024-09-03T07:33:50.700386286+00:00,2024-09-03T06:45:10.808711529+00:00,-0.811081,hours
3,Central 80% volume-duration change,191.597833,189.472012,-1.109523,%
4,Volume-quantile asymmetry change,-0.018958,-0.027331,-0.008373,dimensionless
5,Excess-volume change,634909050.065916,633281011.703299,-0.256421,%


### 1. Peak-time shift

**Question:** does the selected crest occur earlier or later?

$$\Delta t_p=t_{p,c}-t_{p,p}.$$

The peak is the first saved maximum in the shared analysis window. It is not a fitted quadratic peak, a model travel-time parameter, or the historical calibrated feature. We report the change in **hours**, with minutes alongside it.

A +5-minute shift means the child crest occurs five minutes later. It does **not** establish that the centroid or tail moves later. Nearly flat or double crests deserve inspection: in 54, a small change in which lobe is tallest can cause a large peak-time shift without translating the whole hydrograph.

The vertical lines mark the two peak times on one shared clock.

In [5]:
display(pd.Series({"peak shift (minutes)": selected.peak_shift_hours * 60,
                   "peak shift (hours)": selected.peak_shift_hours}))
metric_figure(states, EXAMPLE, CHILD, 'peak_shift_hours').show()

peak shift (minutes)    4.000000
peak shift (hours)      0.066667
dtype: float64

### 2. Peak-excess change

**Question:** does the pulse rise further above the shared minimum baseline?

$$E_p=Q_p-B,\qquad R_H=100\left(\frac{E_{p,c}}{E_{p,p}}-1\right).$$

This percentage describes height above the **same baseline**, not total discharge. Keep total peak change alongside it:

$$R_{Q_p}=100\left(\frac{Q_{p,c}}{Q_{p,p}}-1\right).$$

For example, total peaks 1000 → 1010 m³/s give +1%; above a threshold of 900, excess peaks 100 → 110 give +10%. The absolute height difference is identical; the denominator changes.

Positive means a higher child peak, negative a lower one. Calling this “less/more attenuation” is a relative parent–child description; it is not a separately calculated inlet-to-outlet attenuation ratio. High baseline levels can amplify percentages by making the parent excess small.

In [6]:
display(pd.Series({"peak-excess change (%)": selected.peak_excess_change_pct,
                   "total peak change (%)": selected.total_peak_change_pct,
                   "absolute peak change (m³/s)": child.peak_cms - parent.peak_cms}))
metric_figure(states, EXAMPLE, CHILD, 'peak_excess_change_pct').show()

peak-excess change (%)         0.527603
total peak change (%)          0.298409
absolute peak change (m³/s)    5.588204
dtype: float64

### 3. Excess-volume centroid shift

**Question:** does the centre of the selected event's excess volume pass earlier or later?

$$V=\int_{t_a}^{t_b}e(t)\,dt,\qquad \mu=\frac{\int_{t_a}^{t_b}t\,e(t)\,dt}{V},\qquad \Delta\mu=\mu_c-\mu_p.$$

Every time contributes in proportion to its excess discharge. A long, low tail can therefore move the centroid even when the peak barely moves. This makes the centroid complementary to peak timing.

Each pulse is normalized by **its own excess volume**. Both centroids remain on the same absolute clock; the curves are not aligned before calculating the shift. Positive means later passage of the volume centre. This is neither tracked particle travel time nor the routing operator's mean delay inferred from K and X.

The vertical lines show the two volume centroids.

A small late volume has a long time separation from the original centroid: adding volume $\delta V$ centred at $t_T$ shifts the centroid by $\delta V/(V+\delta V)\,(t_T-\mu)$. Thus small volume does not guarantee a negligible centroid shift.

In [7]:
display(pd.Series({"centroid shift (minutes)": selected.centroid_shift_hours * 60,
                   "half-volume time shift (minutes; supporting)": selected.median_shift_hours * 60}))
metric_figure(states, EXAMPLE, CHILD, 'centroid_shift_hours').show()

centroid shift (minutes)                       -48.664861
half-volume time shift (minutes; supporting)   -22.768034
dtype: float64

### 4. Central 80% volume-duration change

**Question:** is the main body of excess volume more spread out or concentrated in time?

Define $F(t)=V^{-1}\int_{t_a}^{t}e(s)\,ds$. The time $t_f$ satisfies $F(t_f)=f$.

$$D_{80}=t_{90}-t_{10},\qquad R_{D80}=100\left(\frac{D_{80,c}}{D_{80,p}}-1\right).$$

Report both the **percentage change** and the **absolute change in hours**. Positive means a longer central-volume passage interval; negative means a shorter one. A rigid translation of the complete retained distribution gives zero change.

This interval discards the first and last 10% of volume. It is not the entire shared-window duration, and it is not the width at 80% or 50% of peak height. It reduces the influence of extreme tails, but still depends on the baseline and window selection.

The cumulative-volume plot marks the 10% and 90% passages for each pulse. The displayed CDF joins exact cumulative values at saved nodes; the marker positions use analytic within-segment integration.

In [8]:
display(pd.Series({"parent D80 (hours)": parent.duration80_hours,
                   "child D80 (hours)": child.duration80_hours,
                   "change (hours)": selected.duration80_change_hours,
                   "change (%)": selected.duration80_change_pct}))
metric_figure(states, EXAMPLE, CHILD, 'duration80_change_pct').show()

parent D80 (hours)    191.597833
child D80 (hours)     189.472012
change (hours)         -2.125822
change (%)             -1.109523
dtype: float64

### 5. Volume-quantile asymmetry change

**Question:** does the volume distribution become more weighted toward a long late portion or a long early portion?

$$A=\frac{(t_{90}-t_{50})-(t_{50}-t_{10})}{t_{90}-t_{10}},\qquad \Delta A=A_c-A_p.$$

$A>0$ means the late half of the central-volume interval lasts longer than the early half; $A<0$ means the reverse. $A=0$ means these two intervals have equal duration—it does not prove complete hydrograph symmetry.

The response is an **absolute dimensionless difference**, not a percentage. A change from 0.20 to 0.30 is +0.10. Pure translation and uniform time stretching of the retained distribution preserve A; reshaping or truncation may change it. The reference is the volume median, not the peak, so this is not a direct falling-limb/rising-limb ratio.

The bars compare the early and late half-volume durations.

In [9]:
display(pd.Series({"parent A": parent.asymmetry, "child A": child.asymmetry,
                   "change in A": selected.asymmetry_change}))
metric_figure(states, EXAMPLE, CHILD, 'asymmetry_change').show()

parent A      -0.018958
child A       -0.027331
change in A   -0.008373
dtype: float64

### 6. Excess-volume change

**Question:** how much water passes above the shared baseline within the common analysis window?

$$V=\int_{t_a}^{t_b}[Q(t)-B]\,dt,\qquad R_V=100\left(\frac{V_c}{V_p}-1\right).$$

Parent and child use **the same baseline and time interval**. Positive means a larger selected excess volume. A pure translation preserves volume only if the window retains the entire translated contribution; finite boundaries can otherwise change the captured volume.

This is not a complete water-conservation test: it describes a finite selected part of the discharge. With identical baseline and window, $V_c-V_p=\int_{t_a}^{t_b}(Q_c-Q_p)\,dt$. This identity is checked numerically. Adding equal plateau volume to both states leaves this absolute difference unchanged but changes the denominator of the percentage response.

The shaded areas are the selected excess volumes.

In [10]:
display(pd.Series({"parent excess volume (million m³)": parent.volume_m3 / 1e6,
                   "child excess volume (million m³)": child.volume_m3 / 1e6,
                   "change (%)": selected.volume_change_pct}))
metric_figure(states, EXAMPLE, CHILD, 'volume_change_pct').show()

parent excess volume (million m³)    634.909050
child excess volume (million m³)     633.281012
change (%)                            -0.256421
dtype: float64

## 4. Investigate the full response catalogue

Each point below is a comparison against that example's base. Blue circles are partial collapses; orange diamonds are single centerlines. Hover reveals the state aliases. Zero means no change in that particular response.

The six measures are **complementary, not statistically independent**. Avoid reducing their signs to one “faster/slower” label. Window/baseline sensitivity, very flat peaks and 54's lobe selection should be examined before using these as prediction targets.

In [11]:
overview_figure(responses).show()
# Change EXAMPLE above to inspect another network's numerical comparisons.
display(responses.loc[responses.example.eq(EXAMPLE),
    ["state", "aliases", "comparison", *METRICS, "total_peak_change_pct", "duration80_change_hours"]])

,state,aliases,comparison,peak_shift_hours,peak_excess_change_pct,centroid_shift_hours,duration80_change_pct,asymmetry_change,volume_change_pct,total_peak_change_pct,duration80_change_hours
1573,S001_unit_102,S001_unit_102 | island_46_087,partial collapse,0.016667,0.029269,-0.057060,-0.073584,-0.000557,-0.017358,0.016555,-0.140984
1574,S002_unit_112,S002_unit_112 | island_46_063,partial collapse,-0.050000,0.007285,-0.042366,-0.027175,0.000172,-0.017952,0.004120,-0.052067
1575,S003_unit_135,S003_unit_135 | island_46_133,partial collapse,-0.050000,0.008944,-0.042903,-0.028094,0.000166,-0.018129,0.005059,-0.053828
1576,S004_unit_24,S004_unit_24 | island_46_026,partial collapse,0.016667,-0.013937,-0.011704,0.010695,0.000006,0.002484,-0.007883,0.020491
1577,S005_unit_171,S005_unit_171 | island_46_173,partial collapse,-0.016667,0.000162,-0.014403,-0.008498,0.000069,-0.006172,0.000092,-0.016281
...,...,...,...,...,...,...,...,...,...,...,...
1792,island_46_086,island_46_086,partial collapse,-0.250000,0.101299,-0.172100,-0.133398,0.000604,-0.076455,0.057294,-0.255587
1793,island_46_112,island_46_112,partial collapse,-0.116667,-0.026313,-0.014065,0.049042,0.001038,-0.011025,-0.014883,0.093963
1794,island_46_120,island_46_120,partial collapse,-0.066667,-0.000145,-0.044754,-0.025856,0.000224,-0.019212,-0.000082,-0.049540
1795,island_46_149,island_46_149,partial collapse,-0.066667,0.004384,-0.046441,-0.028484,0.000208,-0.019792,0.002480,-0.054574


## 5. Sensitivity to the response window

The primary cutoff is **0.5%**; 1% and 2% use their own shared onset and end, with the same minimum baseline. These calculations update **both boundaries**, unlike the earlier input-start-fixed diagnostic tables.

The table and plots compare all six responses for the selected pair. Inspect changes in magnitude, sign and ranking; small changes near zero should not be interpreted as substantive sign reversals.

In [12]:
sensitivity_table, sensitivity_plot = sensitivity_figure(EXAMPLE, CHILD)
display(sensitivity_table)
sensitivity_plot.show()

,window,peak_shift_hours,peak_excess_change_pct,centroid_shift_hours,duration80_change_pct,asymmetry_change,volume_change_pct
0,0.5%,0.066667,0.527603,-0.811081,-1.109523,-0.008373,-0.256421
1,1%,0.066667,0.527603,-0.725462,-1.032978,-0.007636,-0.234181
2,2%,0.066667,0.527603,-0.585057,-0.893453,-0.006379,-0.183562


## 6. Reproducibility and limits

Current primary and sensitivity tables are in `RAPID/outputs/hydrograph_response/`. The manifest records completed-run files, reviewed window locations, baseline definitions, calculation code and output hashes. The loader refuses stale definitions or tables; plotted outlets are also hash-checked.

Set `RECALCULATE=True` to rebuild from completed routed hydrographs. This does not rerun the model. Historical high-threshold, per-state crossing results are archived in `historical_horizontal_threshold_20261013/` and are not loaded here.

The 0.5% rule is an operational convention, not a unique physical event boundary. All four distribution metrics depend on the retained excess volume and window. Double crests (especially 54), existing observation interpolation and the provisional 21/22 clocks remain relevant limitations.

In [13]:
display(pd.Series({k: manifest[k] for k in ["status", "states", "comparisons", "primary_window", "baseline", "peak", "integration"]}))
print("Current response tables:", OUT)

Current response tables: /Users/6256481/Code/river-hierarchy/RAPID/outputs/hydrograph_response


status            Recalculated completed reruns: primary shared ...
states                                                         2202
comparisons                                                    2187
primary_window                                                 0.5%
baseline          One fixed minimum per example across all compl...
peak              First saved maximum inside the shared analysis...
integration       Exact integrals of piecewise-linear reconstruc...
dtype: object